# 85 — Campaign C queue + S0 loop

82（一覧・promote・resolve）と 83（S0）を **一つのループ**で回す。

- `READY_SHARED` の候補は S0 → archive なら次へ継続
- **`NEED_CANONICAL_M2` / `WAITING_FOR_CANONICAL_M2` で停止** → notebook 73
- `SELECTED` で停止（78 scaffold / production M6 禁止）
- GPU M2・production M6 は起動しない
- `source_drift=True` は exploratory reuse のみ


In [ ]:
from pathlib import Path
import os, sys, json

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'm7_campaign_c_loop.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/m7_campaign_c_loop.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT')
os.environ.setdefault('VALIDATED_RG_M7C_RUN_ID', 'M7-20260720T081500Z-c8d5f02b3c96')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)


In [ ]:
from src.m7_candidate_queue import search_root_for, list_queue_rows
from src.m7_status import M7_RUN_ID_CAMPAIGN_C
from src.rank_sweep import default_sweep_config
from src.runtime import validate_persistent_root
from src.m7_campaign_c_loop import run_campaign_c_queue_s0_loop

validate_persistent_root()
M7C_RUN_ID = os.environ.get('VALIDATED_RG_M7C_RUN_ID', M7_RUN_ID_CAMPAIGN_C)
SEARCH_ROOT = search_root_for(PERSIST_ROOT, M7C_RUN_ID)
MAX_CANDIDATES = int(os.environ.get('VALIDATED_RG_LOOP_MAX', '32'))
SWEEP_CONFIG = default_sweep_config(
    max_factor_rank=int(os.environ.get('VALIDATED_RG_SWEEP_MAX_RANK', '96')),
    rank_grid=[16, 24, 25, 32, 36, 48, 49, 64, 81, 96],
    oversampling=16,
    power_iterations=2,
    seed=20260720,
    engineering_margin=0.05,
    sectors_per_shard=8,
)
ROWS = list_queue_rows(SEARCH_ROOT, persistent_root=PERSIST_ROOT)
print('SEARCH_ROOT', SEARCH_ROOT)
print('MAX_CANDIDATES', MAX_CANDIDATES)
print('queue rows', len(ROWS))


In [ ]:
RESULT = run_campaign_c_queue_s0_loop(
    project_root=PROJECT_ROOT,
    persistent_root=PERSIST_ROOT,
    search_root=SEARCH_ROOT,
    campaign_run_id=M7C_RUN_ID,
    max_candidates=MAX_CANDIDATES,
    sweep_config=SWEEP_CONFIG,
)
print(json.dumps({
    k: RESULT.get(k)
    for k in (
        'series_status', 'processed', 'candidate_id', 'selected_rank',
        'next_notebook', 'env_hint', 'note',
    )
    if k in RESULT
}, indent=2, ensure_ascii=False))
print('events', len(RESULT.get('events') or []))
for ev in (RESULT.get('events') or [])[-6:]:
    if ev.get('event') == 'PREPARE':
        print(' PREPARE', ev.get('candidate_id'), ev.get('state'), ev.get('canonical_run_id'))
    else:
        print(
            ' S0', ev.get('candidate_id'),
            ev.get('series_status') or ev.get('status'),
            ev.get('selection_status'),
        )


In [ ]:
status = RESULT.get('series_status')
if status in {'NEED_CANONICAL_M2', 'WAITING_FOR_CANONICAL_M2', 'NEED_M2'}:
    print('Stop for notebook 73. Paste in Python (not !export):')
    hint = RESULT.get('env_hint') or {}
    print(f"os.environ['VALIDATED_RG_STAGED_CANDIDATE'] = {hint.get('VALIDATED_RG_STAGED_CANDIDATE')!r}")
    print("os.environ.pop('VALIDATED_RG_STAGED_PACKAGE', None)")
    if status == 'WAITING_FOR_CANONICAL_M2':
        print('Do NOT start a second full M2; wait for owner.')
elif status == 'SELECTED':
    print('Stop: exploratory SELECTED. Notebook 78 is still PR-2 scaffold.')
    print('Do NOT start production M6.')
elif status == 'EXHAUSTED':
    print('No actionable candidates left. Need new search (72) or higher j2.')
elif status == 'BUDGET_DONE':
    print('Hit LOOP_MAX; raise VALIDATED_RG_LOOP_MAX and re-run 85.')
else:
    print('series_status', status)
    print(RESULT.get('note'))
